# PARCO for the PVRPWDP

Learning a Parallel AutoRegressive policy for the **Perishable Vehicle Routing Problem with Drones and Pickup (PVRPWDP)**.

**Key Features:**
- ⏰ Time windows constraints
- 🍎 Perishability (freshness) constraints  
- 🔋 Drone battery endurance limits
- 🚚🚁 Heterogeneous fleet (trucks + drones)

This notebook demonstrates training with **epoch-based data loading**.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import torch
from rl4co.utils.trainer import RL4COTrainer

from parco.envs.pvrpwdp import PVRPWDPVEnv
from parco.envs.pvrpwdp.generator import PVRPWDPGenerator
from parco.models import PARCORLModule, PARCOPolicy

# Ensure we're in the project root directory
project_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'examples' else os.getcwd()
os.chdir(project_root)
print(f"Working directory: {os.getcwd()}")

# Device setup - auto-detect CUDA
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")
# if torch.cuda.is_available():
#     print(f"GPU: {torch.cuda.get_device_name(0)}")
#     print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

# Verify data files exist
print(f"\nData files check:")
print(f"  Training epoch 0: {os.path.exists('data/train_data_npz/pvrpwdp_epoch_00.npz')}")
print(f"  Validation: {os.path.exists('data/val_data/val.npz')}")
print(f"  Test: {os.path.exists('data/test_data/test.npz')}")

e:\Đồ án\parco\.venv\Lib\site-packages\torchrl\data\replay_buffers\samplers.py:37: UserWarning: Failed to import torchrl C++ binaries. Some modules (eg, prioritized replay buffers) may not work with your installation. This is likely due to a discrepancy between your package version and the PyTorch version. Make sure both are compatible. Usually, torchrl majors follow the pytorch majors within a few days around the release. For instance, TorchRL 0.5 requires PyTorch 2.4.0, and TorchRL 0.6 requires PyTorch 2.5.0.
  warnings.warn(EXTENSION_WARNING)
e:\Đồ án\parco\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Working directory: e:\Đồ án\parco
Using device: cuda
CUDA available: True

Data files check:
  Training epoch 0: True
  Validation: True
  Test: True


## Environment Setup

**✅ FIXED: batch_size int vs list issue!**

**⚠️ IMPORTANT: Restart kernel và Run All để test fix!**

We create the PVRPWDP environment with pre-generated data:
- **Training data**: `data/train_data_npz/` (18 epoch files: epoch 0-17)
- **Validation data**: `data/val_data/val.npz` (your custom validation set)
- **Test data**: `data/test_data/test.npz` (your custom test set)

All files must be in the `data/` folder as required by RL4CO framework.

In [2]:
# Environment configuration
import os

# Get absolute paths to data directories
project_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'examples' else os.getcwd()
epoch_data_dir = os.path.join(project_root, "data", "train_data_npz")
val_file = os.path.join("val_data", "val.npz")  # RL4CO adds 'data/' prefix
test_file = os.path.join("test_data", "test.npz")  # RL4CO adds 'data/' prefix

print(f"📂 Data directories:")
print(f"   Project root: {project_root}")
print(f"   Epoch data: {epoch_data_dir}")
print(f"   Epoch 0 exists: {os.path.exists(os.path.join(epoch_data_dir, 'pvrpwdp_epoch_00.npz'))}")
print()

env = PVRPWDPVEnv(
    # Epoch data configuration (for training) - USE ABSOLUTE PATH
    epoch_data_dir=epoch_data_dir,
    epoch_file_pattern="pvrpwdp_epoch_{epoch:02d}.npz",
    use_epoch_data=True,
    fallback_to_generator=False,  # ❌ DISABLE FALLBACK - ONLY USE NPZ FILES!
    
    # Standard validation/test files (RL4CO adds 'data/' prefix automatically)
    val_file=val_file,
    test_file=test_file,
    
    # Generator parameters (NOT USED - only for reference)
    generator_params=dict(
        num_loc=20,          # Number of customers (MUST MATCH NPZ DATA)
        num_agents=4,        # Number of vehicles (MUST MATCH NPZ DATA: your data has 4 agents!)
        min_demand=1,
        max_demand=10,
        capacity=40.0,       # Vehicle capacity
        endurance=10.0,      # Drone battery endurance
        speed=1.0,           # Truck speed
        drone_speed=2.0,     # Drone speed (faster than trucks)
        tw_expansion=3.0,    # Time window size factor
        freshness_factor=2.0, # Freshness constraint factor
    )
)

# Print epoch data information
env.print_epoch_data_info()

print("\n⚠️ IMPORTANT:")
print("  - Generator DISABLED (fallback_to_generator=False)")
print("  - Training will ONLY use pre-generated NPZ files")
print("  - Your data has: 20 customers, 4 agents")
print("  - If NPZ file not found or corrupt → training will FAIL with clear error")

📂 Data directories:
   Project root: e:\Đồ án\parco
   Epoch data: e:\Đồ án\parco\data\train_data_npz
   Epoch 0 exists: True


EPOCH DATA CONFIGURATION
Epoch Data Directory: e:\Đồ án\parco\data\train_data_npz
File Pattern:         pvrpwdp_epoch_{epoch:02d}.npz
Use Epoch Data:       True
Fallback to Generator: False
Current Epoch:        0
Max Epochs:           None

Available Epochs:     0


⚠️ IMPORTANT:
  - Generator DISABLED (fallback_to_generator=False)
  - Training will ONLY use pre-generated NPZ files
  - Your data has: 20 customers, 4 agents
  - If NPZ file not found or corrupt → training will FAIL with clear error


# DEBUG


In [ ]:
# 🔍 DEBUG: Check TD values after reset with small batch
print("=" * 80)
print("🔍 DEBUGGING: Inspecting TensorDict after env.reset() with batch_size=2")
print("=" * 80)

from rl4co.data.utils import load_npz_to_tensordict

# Load small batch
td_loaded = load_npz_to_tensordict('data/train_data_npz/pvrpwdp_epoch_00.npz')
td_small = td_loaded[:2]  # Only 2 instances for detailed inspection

print("\n📦 Input TD (from NPZ):")
print(f"  depot shape: {td_small['depot'].shape}")
print(f"  depot values:\n{td_small['depot']}")
print(f"\n  locs shape: {td_small['locs'].shape}")
print(f"  locs[0] (first 3 customers):\n{td_small['locs'][0, :3]}")
print(f"\n  demand shape: {td_small['demand'].shape}")
print(f"  demand values:\n{td_small['demand']}")
print(f"\n  time_windows shape: {td_small['time_windows'].shape}")
print(f"  time_windows[0] (first 3):\n{td_small['time_windows'][0, :3]}")
print(f"\n  waiting_time shape: {td_small['waiting_time'].shape}")
print(f"  waiting_time values:\n{td_small['waiting_time']}")
print(f"\n  agents_speed shape: {td_small['agents_speed'].shape}")
print(f"  agents_speed values:\n{td_small['agents_speed']}")
print(f"\n  agents_capacity shape: {td_small['agents_capacity'].shape}")
print(f"  agents_capacity values:\n{td_small['agents_capacity']}")
print(f"\n  agents_endurance shape: {td_small['agents_endurance'].shape}")
print(f"  agents_endurance values:\n{td_small['agents_endurance']}")

# Reset environment
td_reset = env.reset(td_small)

print("\n" + "=" * 80)
print("✅ After env.reset():")
print("=" * 80)

# Check all TD fields for NaN/Inf
import torch

def check_tensor(name, tensor):
    has_nan = torch.isnan(tensor).any().item()
    has_inf = torch.isinf(tensor).any().item()
    min_val = tensor.min().item()
    max_val = tensor.max().item()
    status = "❌ NaN!" if has_nan else ("⚠️ Inf!" if has_inf else "✅ OK")
    print(f"  {name:25s} {status:10s} shape: {str(tensor.shape):20s} range: [{min_val:10.2f}, {max_val:10.2f}]")
    return has_nan or has_inf

print("\n🔍 Checking all TD fields:")
has_issues = False
for key in sorted(td_reset.keys()):
    if isinstance(td_reset[key], torch.Tensor):
        has_issues |= check_tensor(key, td_reset[key])

print("\n📊 Detailed values (batch 0):")
print(f"\n  max_time: {td_reset['max_time']}")
print(f"\n  trip_deadline:\n{td_reset['trip_deadline']}")
print(f"\n  waiting_time shape: {td_reset['waiting_time'].shape}")
print(f"  waiting_time[0] (depot + first 3 customers):")
print(f"    Depot (0-3):    {td_reset['waiting_time'][0, :4]}")
print(f"    Customers (4-6): {td_reset['waiting_time'][0, 4:7]}")
print(f"    Last 3:          {td_reset['waiting_time'][0, -3:]}")

print(f"\n  time_window shape: {td_reset['time_window'].shape}")
print(f"  time_window[0] (depot + first 3):")
print(f"    Depot (0-3):\n{td_reset['time_window'][0, :4]}")
print(f"    Customers (4-6):\n{td_reset['time_window'][0, 4:7]}")

print(f"\n  agents_endurance: {td_reset['agents_endurance']}")
print(f"  agents_speed: {td_reset['agents_speed']}")
print(f"  agents_capacity: {td_reset['agents_capacity']}")

print(f"\n  current_time: {td_reset['current_time']}")
print(f"  used_endurance: {td_reset['used_endurance']}")
print(f"  used_capacity: {td_reset['used_capacity']}")

if has_issues:
    print("\n❌ FOUND NaN/Inf in TD after reset!")
else:
    print("\n✅ All TD fields are valid (no NaN/Inf)")

print("\n" + "=" * 80)

In [ ]:
# 🔍 DEBUG: Test embedding computation to find where NaN appears
print("=" * 80)
print("🔍 DEBUGGING: Testing Embedding Layers")
print("=" * 80)

from parco.models.env_embeddings.pvrpwdp import PVRPWDPInitEmbedding

# Create embedding layer directly (simpler than going through policy)
init_embedding = PVRPWDPInitEmbedding(
    embed_dim=128,
    use_polar_feats=True,
    normalize_endurance_by_max=False
)

print(f"\n📦 Init Embedding Module: {type(init_embedding).__name__}")

# Test with our small batch
print("\n🧪 Testing init_embedding with td_reset...")

try:
    with torch.no_grad():
        # Call init embedding
        embeddings = init_embedding(td_reset)
    
    print(f"\n✅ Init embedding successful!")
    print(f"   Output shape: {embeddings.shape}")
    print(f"   Output dtype: {embeddings.dtype}")
    
    # Check for NaN/Inf
    has_nan = torch.isnan(embeddings).any().item()
    has_inf = torch.isinf(embeddings).any().item()
    
    if has_nan:
        print(f"\n❌ FOUND NaN in embeddings!")
        nan_count = torch.isnan(embeddings).sum().item()
        print(f"   NaN count: {nan_count} / {embeddings.numel()}")
        
        # Find where NaN appears
        nan_mask = torch.isnan(embeddings)
        nan_indices = torch.nonzero(nan_mask, as_tuple=False)
        print(f"   First 10 NaN locations: {nan_indices[:10]}")
    elif has_inf:
        print(f"\n⚠️  FOUND Inf in embeddings!")
        inf_count = torch.isinf(embeddings).sum().item()
        print(f"   Inf count: {inf_count} / {embeddings.numel()}")
    else:
        print(f"\n✅ No NaN/Inf in embeddings!")
        print(f"   Min: {embeddings.min().item():.4f}")
        print(f"   Max: {embeddings.max().item():.4f}")
        print(f"   Mean: {embeddings.mean().item():.4f}")
        print(f"   Std: {embeddings.std().item():.4f}")
        
        # Show sample values
        print(f"\n   Sample embeddings[0, 0, :10]:")
        print(f"   {embeddings[0, 0, :10]}")
        
except Exception as e:
    print(f"\n❌ ERROR in init_embedding: {e}")
    import traceback
    traceback.print_exc()

# Check normalization scalers
print("\n" + "=" * 80)
print("🔍 Checking Normalization Scalers:")
print("=" * 80)

try:
    # Get scalers from embedding module
    print(f"\n📊 Computing scalers from td_reset...")
    
    num_agents = td_reset["action_mask"].shape[-2]
    
    # Demand scaler
    demands = td_reset["demand"][..., 0, num_agents:]
    capacities = td_reset["agents_capacity"]
    demand_scaler = torch.maximum(
        demands.max(dim=-1, keepdim=True)[0],
        capacities.max(dim=-1, keepdim=True)[0]
    ).unsqueeze(-1)
    print(f"   demand_scaler: {demand_scaler[0].item():.4f}")
    
    # Speed scaler (from agents_speed)
    speed_scaler = td_reset["agents_speed"].max(dim=-1, keepdim=True)[0].unsqueeze(-1)
    print(f"   speed_scaler: {speed_scaler[0].item():.4f}")
    
    # Time scaler (from time_window latest)
    time_scaler = td_reset["time_window"][..., num_agents:, 1].max(dim=-1, keepdim=True)[0].unsqueeze(-1)
    print(f"   time_scaler: {time_scaler[0].item():.4f}")
    
    # Endurance scaler
    endurance_scaler = td_reset["agents_endurance"].max(dim=-1, keepdim=True)[0].unsqueeze(-1)
    print(f"   endurance_scaler (max endurance): {endurance_scaler[0].item():.4f}")
    print(f"   Using time_scaler for normalization: {time_scaler[0].item():.4f}")
    
    # Check normalized values
    print(f"\n🔢 Normalized values:")
    print(f"   waiting_time / time_scaler:")
    waiting_norm = td_reset["waiting_time"] / time_scaler
    print(f"     Min: {waiting_norm.min().item():.4f}")
    print(f"     Max: {waiting_norm.max().item():.4f}")
    print(f"     Depot: {waiting_norm[0, :4]}")
    print(f"     Customer: {waiting_norm[0, 4:7]}")
    
    print(f"\n   agents_endurance / time_scaler:")
    endurance_norm = td_reset["agents_endurance"][..., None] / time_scaler
    print(f"     Min: {endurance_norm.min().item():.4f}")
    print(f"     Max: {endurance_norm.max().item():.4f}")
    print(f"     Values[0]: {endurance_norm[0].squeeze()}")
    
    # Check if any scaler is 0 or too small
    if demand_scaler[0] < 1e-6:
        print(f"\n⚠️  WARNING: demand_scaler too small: {demand_scaler[0].item()}")
    if speed_scaler[0] < 1e-6:
        print(f"\n⚠️  WARNING: speed_scaler too small: {speed_scaler[0].item()}")
    if time_scaler[0] < 1e-6:
        print(f"\n⚠️  WARNING: time_scaler too small: {time_scaler[0].item()}")
        
except Exception as e:
    print(f"\n❌ ERROR checking scalers: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "=" * 80)

In [ ]:
# 🔍 DEBUG: Check location coordinates normalization
print("=" * 80)
print("🔍 DEBUGGING: Location Coordinates Normalization")
print("=" * 80)

print("\n📍 Original locations from NPZ (td_small):")
print(f"  depot: {td_small['depot'][0]}")
print(f"  locs[0] range:")
print(f"    X: [{td_small['locs'][0, :, 0].min().item():.2f}, {td_small['locs'][0, :, 0].max().item():.2f}]")
print(f"    Y: [{td_small['locs'][0, :, 1].min().item():.2f}, {td_small['locs'][0, :, 1].max().item():.2f}]")
print(f"\n  Sample locs[0] (first 5 customers):")
for i in range(5):
    print(f"    Customer {i}: ({td_small['locs'][0, i, 0].item():8.2f}, {td_small['locs'][0, i, 1].item():8.2f})")

print("\n📍 After env.reset() - td_reset['locs'] (depot + customers):")
print(f"  Shape: {td_reset['locs'].shape}  # Should be [B, num_agents + N, 2]")
print(f"  locs[0] range:")
print(f"    X: [{td_reset['locs'][0, :, 0].min().item():.2f}, {td_reset['locs'][0, :, 0].max().item():.2f}]")
print(f"    Y: [{td_reset['locs'][0, :, 1].min().item():.2f}, {td_reset['locs'][0, :, 1].max().item():.2f}]")

print(f"\n  Depot locations (first 4 entries = agents):")
for i in range(4):
    print(f"    Agent {i} depot: ({td_reset['locs'][0, i, 0].item():8.2f}, {td_reset['locs'][0, i, 1].item():8.2f})")

print(f"\n  Customer locations (entries 4+):")
for i in range(4, 9):  # Show first 5 customers
    print(f"    Customer {i-4}: ({td_reset['locs'][0, i, 0].item():8.2f}, {td_reset['locs'][0, i, 1].item():8.2f})")

# Check if coordinates have negative values
has_negative_x = (td_reset['locs'][:, :, 0] < 0).any().item()
has_negative_y = (td_reset['locs'][:, :, 1] < 0).any().item()

if has_negative_x or has_negative_y:
    print(f"\n⚠️  FOUND NEGATIVE COORDINATES!")
    print(f"   Negative X: {has_negative_x}")
    print(f"   Negative Y: {has_negative_y}")
    print(f"\n   This means coordinates are NOT normalized to [0, 1]!")
    print(f"   They are in ABSOLUTE coordinates (meters/km)")
else:
    print(f"\n✅ All coordinates are non-negative (normalized)")

# Check coordinate scale
max_coord = max(
    abs(td_reset['locs'][:, :, 0].max().item()),
    abs(td_reset['locs'][:, :, 0].min().item()),
    abs(td_reset['locs'][:, :, 1].max().item()),
    abs(td_reset['locs'][:, :, 1].min().item())
)

print(f"\n📏 Coordinate scale:")
print(f"   Max absolute coordinate: {max_coord:.2f}")
if max_coord <= 1.0:
    print(f"   ✅ Coordinates normalized to [0, 1] range")
elif max_coord <= 100:
    print(f"   ⚠️  Coordinates in normalized scale (~0-100)")
elif max_coord <= 1000:
    print(f"   ⚠️  Coordinates in meters scale (~0-1000)")
else:
    print(f"   ⚠️  Coordinates in meters/km scale (LARGE VALUES!)")
    print(f"   This may cause numerical instability in embeddings!")

# Compute distances for scale reference
print(f"\n📐 Distance scale check:")
depot = td_reset['locs'][0, 0]  # First agent's depot
customer_0 = td_reset['locs'][0, 4]  # First customer
dist = torch.norm(customer_0 - depot, p=2).item()
print(f"   Distance from depot to customer 0: {dist:.2f}")

# Calculate max distance
all_dists = torch.norm(td_reset['locs'][0, 4:] - depot.unsqueeze(0), p=2, dim=-1)
max_dist = all_dists.max().item()
print(f"   Max distance from depot: {max_dist:.2f}")

# Expected travel time
speed_min = td_reset['agents_speed'][0].min().item()
travel_time_max = max_dist / speed_min
print(f"   Max travel time (dist / min_speed): {travel_time_max:.2f}s")
print(f"   Compare to max_time: {td_reset['max_time'][0].item():.2f}s")

print("\n" + "=" * 80)

In [ ]:
# 🔍 DEBUG: Test normalized embeddings directly
print("=" * 80)
print("🔍 DEBUGGING: Testing Normalized Embeddings")
print("=" * 80)

from parco.models.env_embeddings.pvrpwdp import PVRPWDPInitEmbedding

# Create embedding layer
init_emb = PVRPWDPInitEmbedding(
    embed_dim=128,
    use_polar_feats=True,
    normalize_endurance_by_max=False
)

print("\n🧪 Testing init_embedding.forward(td_reset)...")

try:
    with torch.no_grad():
        embeddings = init_emb(td_reset)
    
    print(f"✅ Embedding computation successful!")
    print(f"   Output shape: {embeddings.shape}")
    
    # Check for NaN/Inf
    has_nan = torch.isnan(embeddings).any().item()
    has_inf = torch.isinf(embeddings).any().item()
    
    if has_nan:
        print(f"\n❌ FOUND NaN in embeddings!")
        nan_mask = torch.isnan(embeddings)
        nan_count = nan_mask.sum().item()
        print(f"   NaN count: {nan_count} / {embeddings.numel()}")
        
        # Find which position has NaN
        nan_indices = torch.nonzero(nan_mask, as_tuple=False)
        print(f"   First NaN at: batch={nan_indices[0, 0]}, pos={nan_indices[0, 1]}, dim={nan_indices[0, 2]}")
        
    elif has_inf:
        print(f"\n⚠️  FOUND Inf in embeddings!")
        inf_count = torch.isinf(embeddings).sum().item()
        print(f"   Inf count: {inf_count} / {embeddings.numel()}")
    else:
        print(f"\n✅ No NaN/Inf in embeddings!")
        print(f"   Min: {embeddings.min().item():.4f}")
        print(f"   Max: {embeddings.max().item():.4f}")
        print(f"   Mean: {embeddings.mean().item():.4f}")
        print(f"   Std: {embeddings.std().item():.4f}")
        
        # Show sample embedding
        print(f"\n   Sample embeddings[0, 0, :5]: {embeddings[0, 0, :5]}")
        print(f"   Sample embeddings[0, 4, :5]: {embeddings[0, 4, :5]}")
    
    # Debug: Check intermediate normalized values
    print("\n📊 Checking normalized coordinates:")
    num_agents = td_reset["action_mask"].shape[-2]
    
    all_locs = td_reset["locs"]
    loc_min = all_locs.min(dim=-2, keepdim=True)[0]
    loc_max = all_locs.max(dim=-2, keepdim=True)[0]
    loc_range = loc_max - loc_min
    
    print(f"   loc_min[0]: {loc_min[0, 0]}")
    print(f"   loc_max[0]: {loc_max[0, 0]}")
    print(f"   loc_range[0]: {loc_range[0, 0]}")
    
    clients_locs = td_reset["locs"][..., num_agents:, :]
    clients_locs_norm = (clients_locs - loc_min) / loc_range
    
    print(f"\n   clients_locs_norm[0, 0]: {clients_locs_norm[0, 0]}")
    print(f"   clients_locs_norm[0] range X: [{clients_locs_norm[0, :, 0].min().item():.4f}, {clients_locs_norm[0, :, 0].max().item():.4f}]")
    print(f"   clients_locs_norm[0] range Y: [{clients_locs_norm[0, :, 1].min().item():.4f}, {clients_locs_norm[0, :, 1].max().item():.4f}]")
    
    # Check if all normalized values are in [0, 1]
    all_in_range = (clients_locs_norm >= 0).all() and (clients_locs_norm <= 1).all()
    if all_in_range:
        print(f"\n   ✅ All normalized coordinates in [0, 1]")
    else:
        print(f"\n   ❌ Some normalized coordinates outside [0, 1]!")
        print(f"      Min: {clients_locs_norm.min().item():.4f}")
        print(f"      Max: {clients_locs_norm.max().item():.4f}")
    
except Exception as e:
    print(f"\n❌ ERROR: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "=" * 80)

In [ ]:
# 🔍 DEBUG: Test full forward pass (encoder + decoder) to find NaN source
print("=" * 80)
print("🔍 DEBUGGING: Testing Full Forward Pass (Encoder + Decoder)")
print("=" * 80)

# Load validation data
from rl4co.data.utils import load_npz_to_tensordict
val_td = load_npz_to_tensordict('data/val_data/val.npz')
val_small = val_td[:2]  # Small batch for testing

print(f"\n📦 Validation data loaded:")
print(f"   Batch size: {val_small.batch_size}")
print(f"   locs shape: {val_small['locs'].shape}")
print(f"   locs range: X=[{val_small['locs'][..., 0].min():.2f}, {val_small['locs'][..., 0].max():.2f}], Y=[{val_small['locs'][..., 1].min():.2f}, {val_small['locs'][..., 1].max():.2f}]")

# Reset environment
val_reset = env.reset(val_small)

print(f"\n✅ Environment reset successful")
print(f"   Reset locs shape: {val_reset['locs'].shape}")

# Test policy forward pass
print(f"\n🧪 Testing policy forward pass...")

try:
    with torch.no_grad():
        # Create fresh policy for testing
        test_policy = PARCOPolicy(
            env_name=env.name,
            embed_dim=128,
            agent_handler="highprob",
            normalization="rms",
            context_embedding_kwargs={
                "normalization": "rms",
                "norm_after": False,
                "normalize_endurance_by_max": False,
                "use_time_to_depot": True,
            },
            norm_after=False,
        )
        
        # Run one decoding step
        out = test_policy(
            val_reset.clone(),
            env,
            phase="val",
            decode_type="greedy",
            return_actions=False,
            calc_reward=False,
        )
    
    print(f"\n✅ Forward pass successful!")
    print(f"   Output keys: {list(out.keys())}")
    
    # Check for NaN in outputs
    has_nan = False
    for key, value in out.items():
        if isinstance(value, torch.Tensor):
            if torch.isnan(value).any():
                print(f"\n❌ FOUND NaN in output['{key}']!")
                nan_count = torch.isnan(value).sum().item()
                print(f"   NaN count: {nan_count} / {value.numel()}")
                has_nan = True
    
    if not has_nan:
        print(f"\n✅ No NaN in policy outputs!")
        if 'reward' in out:
            print(f"   Reward: {out['reward'].mean().item():.4f}")
    
except AssertionError as e:
    print(f"\n❌ AssertionError: {e}")
    print(f"\n   This is the NaN error we're trying to fix!")
    print(f"\n   Let's check intermediate values...")
    
    # Debug: Check init embeddings
    print(f"\n📊 Testing init_embedding on validation data:")
    from parco.models.env_embeddings.pvrpwdp import PVRPWDPInitEmbedding
    
    init_emb = PVRPWDPInitEmbedding(
        embed_dim=128,
        use_polar_feats=True,
        normalize_endurance_by_max=False
    )
    
    with torch.no_grad():
        embeddings = init_emb(val_reset)
    
    has_nan_emb = torch.isnan(embeddings).any().item()
    if has_nan_emb:
        print(f"   ❌ NaN in init_embedding!")
    else:
        print(f"   ✅ init_embedding OK: range=[{embeddings.min():.4f}, {embeddings.max():.4f}]")
    
    # Debug: Check if there's an issue with context embedding
    print(f"\n📊 Checking context embedding computation:")
    from parco.models.env_embeddings.pvrpwdp import PVRPWDPContextEmbedding
    
    context_emb = PVRPWDPContextEmbedding(
        embed_dim=128,
        normalize_endurance_by_max=False,
        use_time_to_depot=True,
    )
    
    # Simulate one step
    val_step = val_reset.clone()
    val_step['current_node'] = torch.zeros(2, 4, dtype=torch.long)  # All agents at depot
    
    try:
        with torch.no_grad():
            num_agents = val_step["action_mask"].shape[-2]
            num_cities = val_step["action_mask"].shape[-1] - num_agents
            context_feats = context_emb._agent_state_embedding(embeddings, val_step, num_agents, num_cities)
        
        has_nan_ctx = torch.isnan(context_feats).any().item()
        if has_nan_ctx:
            print(f"   ❌ NaN in context_embedding!")
            
            # Check individual components
            print(f"\n   Checking context embedding components:")
            print(f"     current_time: {val_step['current_time']}")
            print(f"     agents_speed: {val_step['agents_speed']}")
            print(f"     locs[0, 0]: {val_step['locs'][0, 0]}")  # Depot
            print(f"     current_node: {val_step['current_node']}")
            
            # Check if gathering current location works
            from rl4co.utils.ops import gather_by_index
            cur_loc = gather_by_index(val_step["locs"], val_step["current_node"])
            print(f"     cur_loc: {cur_loc}")
            print(f"     cur_loc has NaN: {torch.isnan(cur_loc).any()}")
            
        else:
            print(f"   ✅ context_embedding OK: range=[{context_feats.min():.4f}, {context_feats.max():.4f}]")
    except Exception as ctx_error:
        print(f"   ❌ Error in context_embedding: {ctx_error}")
        import traceback
        traceback.print_exc()
        
except Exception as e:
    print(f"\n❌ ERROR: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "=" * 80)

In [ ]:
# 🔍 DEBUG: Deep dive into decoder computation
print("=" * 80)
print("🔍 DEBUGGING: Decoder Computation Step-by-Step")
print("=" * 80)

# Use validation data
val_test = val_reset.clone()

print(f"\n📦 Input TD:")
print(f"   locs shape: {val_test['locs'].shape}")
print(f"   locs has NaN: {torch.isnan(val_test['locs']).any()}")
print(f"   locs range: X=[{val_test['locs'][..., 0].min():.2f}, {val_test['locs'][..., 0].max():.2f}], Y=[{val_test['locs'][..., 1].min():.2f}, {val_test['locs'][..., 1].max():.2f}]")

try:
    with torch.no_grad():
        # Create policy
        test_policy = PARCOPolicy(
            env_name=env.name,
            embed_dim=128,
            agent_handler="highprob",
            normalization="rms",
            context_embedding_kwargs={
                "normalization": "rms",
                "norm_after": False,
                "normalize_endurance_by_max": False,
                "use_time_to_depot": True,
            },
            norm_after=False,
        )
        
        print(f"\n🔧 Step 1: Encoder forward pass...")
        # Encoder
        hidden, cached = test_policy.encoder(val_test)
        print(f"   ✅ Encoder done")
        print(f"   hidden shape: {hidden.shape}")
        print(f"   hidden has NaN: {torch.isnan(hidden).any()}")
        print(f"   hidden range: [{hidden.min():.4f}, {hidden.max():.4f}]")
        
        print(f"\n🔧 Step 2: Compute decoder Q (agent context)...")
        # Compute Q (agent embedding for attention)
        glimpse_q = test_policy.decoder._compute_q(cached, val_test)
        print(f"   ✅ glimpse_q computed")
        print(f"   glimpse_q shape: {glimpse_q.shape}")
        print(f"   glimpse_q has NaN: {torch.isnan(glimpse_q).any()}")
        if torch.isnan(glimpse_q).any():
            print(f"   ❌ FOUND NaN in glimpse_q!")
            nan_mask = torch.isnan(glimpse_q)
            nan_indices = torch.nonzero(nan_mask, as_tuple=False)
            print(f"   First NaN at: {nan_indices[0]}")
        else:
            print(f"   glimpse_q range: [{glimpse_q.min():.4f}, {glimpse_q.max():.4f}]")
        
        print(f"\n🔧 Step 3: Compute decoder K, V, L...")
        # Compute K, V, L
        glimpse_k, glimpse_v, logit_k = test_policy.decoder._compute_kvl(cached, val_test)
        print(f"   ✅ glimpse_k, glimpse_v, logit_k computed")
        
        print(f"   glimpse_k shape: {glimpse_k.shape}")
        print(f"   glimpse_k has NaN: {torch.isnan(glimpse_k).any()}")
        if not torch.isnan(glimpse_k).any():
            print(f"   glimpse_k range: [{glimpse_k.min():.4f}, {glimpse_k.max():.4f}]")
        
        print(f"\n   glimpse_v shape: {glimpse_v.shape}")
        print(f"   glimpse_v has NaN: {torch.isnan(glimpse_v).any()}")
        if not torch.isnan(glimpse_v).any():
            print(f"   glimpse_v range: [{glimpse_v.min():.4f}, {glimpse_v.max():.4f}]")
        
        print(f"\n   logit_k shape: {logit_k.shape}")
        print(f"   logit_k has NaN: {torch.isnan(logit_k).any()}")
        if not torch.isnan(logit_k).any():
            print(f"   logit_k range: [{logit_k.min():.4f}, {logit_k.max():.4f}]")
        else:
            print(f"   ❌ FOUND NaN in logit_k!")
        
        print(f"\n🔧 Step 4: Compute attention logits...")
        mask = val_test["action_mask"]
        print(f"   mask shape: {mask.shape}")
        print(f"   mask sum: {mask.sum()}")
        
        # Try to compute logits
        import math
        from torch import bmm
        
        # Simplified pointer attention computation
        print(f"\n   Computing glimpse = softmax(Q @ K^T) @ V...")
        compat = bmm(glimpse_q, glimpse_k.transpose(-2, -1)) / math.sqrt(glimpse_q.size(-1))
        print(f"   compat shape: {compat.shape}")
        print(f"   compat has NaN: {torch.isnan(compat).any()}")
        if not torch.isnan(compat).any():
            print(f"   compat range: [{compat.min():.4f}, {compat.max():.4f}]")
        else:
            print(f"   ❌ NaN in compatibility scores!")
            
        attn = torch.softmax(compat, dim=-1)
        print(f"   attn has NaN: {torch.isnan(attn).any()}")
        if not torch.isnan(attn).any():
            print(f"   attn range: [{attn.min():.4f}, {attn.max():.4f}]")
        else:
            print(f"   ❌ NaN in attention weights!")
            
        glimpse = bmm(attn, glimpse_v)
        print(f"   glimpse shape: {glimpse.shape}")
        print(f"   glimpse has NaN: {torch.isnan(glimpse).any()}")
        if not torch.isnan(glimpse).any():
            print(f"   glimpse range: [{glimpse.min():.4f}, {glimpse.max():.4f}]")
        else:
            print(f"   ❌ NaN in glimpse!")
        
        print(f"\n   Computing final logits = glimpse @ logit_k^T...")
        logits = bmm(glimpse, logit_k.squeeze(-2).transpose(-2, -1)).squeeze(-2) / math.sqrt(glimpse.size(-1))
        print(f"   logits shape: {logits.shape}")
        print(f"   logits has NaN: {torch.isnan(logits).any()}")
        if torch.isnan(logits).any():
            print(f"   ❌ FOUND NaN in final logits!")
            nan_mask = torch.isnan(logits)
            nan_count = nan_mask.sum().item()
            print(f"   NaN count: {nan_count} / {logits.numel()}")
            nan_indices = torch.nonzero(nan_mask, as_tuple=False)
            print(f"   First 5 NaN locations: {nan_indices[:5]}")
        else:
            print(f"   ✅ No NaN in logits!")
            print(f"   logits range: [{logits.min():.4f}, {logits.max():.4f}]")
        
except Exception as e:
    print(f"\n❌ ERROR: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "=" * 80)

In [ ]:
# Force kernel restart to pick up all code changes
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

# Model Configuration

Configure the PARCO policy and RL module with PVRPWDP-specific settings.

## 🎯 Dynamic Resample Strategy

**Enabled by default** with `use_dynamic_resample=True`:

- **Num agents** is **dynamically determined** based on problem size during training
- **Num customers** is randomly sampled from 20 to the NPZ batch size
- Agent count ranges:
  - **≤40 customers**: 4-6 agents (small problems)
  - **41-80 customers**: 6-10 agents (medium problems)  
  - **>80 customers**: 8-12 agents (large problems)

This enables **curriculum learning** with variable-size instances from multiple NPZ files!

## 📋 Configuration Details

- **embed_dim**: 128 (embedding dimension)
- **num_augment**: 16 (SymNCO augmentations as baseline)
- **agent_handler**: "highprob" (select agent with highest probability)
- **Context embedding**: Includes time, capacity, endurance, and deadline features

In [ ]:
emb_dim = 128

# Policy: Neural network for decision making
policy = PARCOPolicy(
    env_name=env.name,
    embed_dim=emb_dim,
    agent_handler="highprob",
    normalization="rms",
    context_embedding_kwargs={
        "normalization": "rms",
        "norm_after": False,
        "normalize_endurance_by_max": False,  # Normalize by time_scaler for consistency
        "use_time_to_depot": True,            # Include time-to-depot and time-to-deadline
    },
    norm_after=False,
)

# RL Module: Policy + Environment + Training algorithm
model = PARCORLModule(
    env, 
    policy=policy,
    train_data_size=1000,       # Instances per epoch (overridden by epoch files)
    val_data_size=100,          # Validation set size
    batch_size=32,              # Training batch size
    num_augment=16,              # SymNCO augmentations for shared baseline
    train_min_agents=4,         # For static mode (not used with dynamic_resample)
    train_max_agents=4,         # For static mode (not used with dynamic_resample)
    train_min_size=20,          # For static mode (not used with dynamic_resample)
    train_max_size=20,          # For static mode (not used with dynamic_resample)
    use_dynamic_resample=True,  # ✅ Enable dynamic resample strategy
    optimizer_kwargs={
        'lr': 1e-4,             # Learning rate
        'weight_decay': 0
    },
)

print("✅ Model created successfully with DYNAMIC RESAMPLE enabled!")
print(f"   Policy parameters: {sum(p.numel() for p in policy.parameters()):,}")
print(f"   Resample strategy: DYNAMIC (num_locs from NPZ → num_agents determined by size)")
print(f"   - num_locs ≤ 40: agents 4-6")
print(f"   - 40 < num_locs ≤ 80: agents 6-10")
print(f"   - num_locs > 80: agents 8-12")

✅ Model created successfully!
   Policy parameters: 991,489
   Configured for: 20 customers, 4 agents


e:\Đồ án\parco\.venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:209: Attribute 'env' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['env'])`.
e:\Đồ án\parco\.venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:209: Attribute 'policy' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['policy'])`.


## Training

Train the model using RL4CO Trainer with your custom data. 

**Note**: 
- Training: 18 epoch files in `data/train_data_npz/` (epochs 0-17)
- Validation: `data/val_data/val.npz`
- Test: `data/test_data/test.npz`
- Generator disabled - using only your pre-generated problem instances

In [4]:
trainer = RL4COTrainer(
    max_epochs=18,           # Number of training epochs (matching available data)
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,               # Number of devices
    logger=None,             # Set to wandb/tensorboard for logging
    gradient_clip_val=1.0,   # Gradient clipping
)

print("🚀 Starting training...")
print(f"   Max epochs: {trainer.max_epochs}")
print(f"   Accelerator: {'GPU (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'CPU'}")

Using 16bit Automatic Mixed Precision (AMP)
Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


🚀 Starting training...
   Max epochs: 18
   Accelerator: GPU (NVIDIA GeForce RTX 3050 Laptop GPU)


e:\Đồ án\parco\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\logger_connector\logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default


In [5]:
# Start training
trainer.fit(model)

print("\n✅ Training completed!")

Loaded data batch size 2000 does not match requested batch size 1000. Using loaded size.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name            | Type        | Params | Mode 
--------------------------------------------------------
0 | env             | PVRPWDPVEnv | 0      | train
1 | policy          | PARCOPolicy | 991 K  | train
2 | baseline        | NoBaseline  | 0      | train
3 | projection_head | MLP         | 33.0 K | train
--------------------------------------------------------
1.0 M     Trainable params
0         Non-trainable params
1.0 M     Total params
4.098     Total estimated model params size (MB)
91        Modules in train mode
0         Modules in eval mode


Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

e:\Đồ án\parco\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
e:\Đồ án\parco\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
e:\Đồ án\parco\.venv\Lib\site-packages\torch\nn\functional.py:2954: UserWarning: Mismatch dtype between input and weight: input dtype = struct c10::Half, weight dtype = float, Cannot dispatch to fused implementation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\layer_norm.cpp:347.)
  return torch.rms_norm(input, normalized_shape, weight, eps)


e:\Đồ án\parco\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Epoch 0: 100%|██████████| 63/63 [03:11<00:00,  0.33it/s, v_num=37, train/reward=-2.46, train/loss=0.581, val/reward=-2.00]

...

Epoch 16: 100%|██████████| 63/63 [02:58<00:00,  0.35it/s, v_num=37, train/reward=-1.42, train/loss=0.0178, val/reward=-1.31]  

Loaded data batch size 2000 does not match requested batch size 1000. Using loaded size.


Epoch 17: 100%|██████████| 63/63 [02:59<00:00,  0.35it/s, v_num=37, train/reward=-1.17, train/loss=0.0731, val/reward=-1.15]  

`Trainer.fit` stopped: `max_epochs=18` reached.


Epoch 17: 100%|██████████| 63/63 [02:59<00:00,  0.35it/s, v_num=37, train/reward=-1.17, train/loss=0.0731, val/reward=-1.15]

✅ Training completed!


In [ ]:
# Test on test.npz with greedy decoding
print("=" * 80)
print("🧪 TESTING: Evaluating Model on Test Set with Greedy Decoding")
print("=" * 80)

from rl4co.data.utils import load_npz_to_tensordict
import pandas as pd

# Load test data
print("\n📦 Loading test data...")
test_td = load_npz_to_tensordict('data/test_data/test.npz')
print(f"   Test set size: {test_td.batch_size} instances")

# Reset environment with test data
print("\n🔄 Resetting environment with test data...")
test_reset = env.reset(test_td)
print(f"   Environment reset successful")

# Run model with greedy decoding to get rewards
print("\n🎯 Running model with greedy decoding...")
with torch.no_grad():
    # Use model.policy() to run inference with custom parameters
    out = model.policy(
        test_reset.clone(),
        env,
        phase="test",
        decode_type="sampling",
        return_init_embeds=False,
    )

# Get rewards
rewards = out['reward']  # [batch_size]
print(f"   ✅ Model inference complete!")
print(f"   Rewards shape: {rewards.shape}")
print(f"   Rewards dtype: {rewards.dtype}")
print(f"   Rewards sample: {rewards[:5]}")

# Extract unvisited count
# Reward is -num_unvisited (penalty for each unvisited customer)
num_customers = 20
num_instances = rewards.shape[0]

# Convert to float first, then to long
unvisited_counts = (-rewards.float()).long()  # Convert to float, negate, then to int
visited_counts = num_customers - unvisited_counts

# Create statistics table
print("\n" + "=" * 80)
print("📊 TEST SET STATISTICS")
print("=" * 80)

# Count instances by number of unvisited customers
unvisited_counts_cpu = unvisited_counts.cpu().numpy()
stats_dict = {}

for unvisited in range(0, num_customers + 1):
    count = (unvisited_counts_cpu == unvisited).sum()
    if count > 0:
        stats_dict[unvisited] = count

# Create dataframe
df_stats = pd.DataFrame([
    {
        'Unvisited Customers': unvisited,
        'Number of Instances': count,
        'Percentage': f"{(count / num_instances) * 100:.1f}%"
    }
    for unvisited, count in sorted(stats_dict.items())
])

print(f"\n{df_stats.to_string(index=False)}")

# Summary statistics
print("\n" + "-" * 80)
print("📈 SUMMARY:")
print("-" * 80)

perfect_instances = (unvisited_counts == 0).sum().item()
print(f"  ✅ Perfect (0 unvisited): {perfect_instances}/{num_instances} ({(perfect_instances/num_instances)*100:.1f}%)")

avg_unvisited = unvisited_counts.float().mean().item()
print(f"  📊 Average unvisited: {avg_unvisited:.2f}")

avg_visited = visited_counts.float().mean().item()
print(f"  📊 Average visited: {avg_visited:.2f}/{num_customers}")

std_unvisited = unvisited_counts.float().std().item()
print(f"  📊 Std dev (unvisited): {std_unvisited:.2f}")

print(f"\n  Min unvisited: {unvisited_counts.min().item()}")
print(f"  Max unvisited: {unvisited_counts.max().item()}")

print("\n" + "=" * 80)